In [25]:
import matplotlib.pyplot as plt
import numpy as np

from build_panel import load_panel_acc

In [26]:
acc_data = load_panel_acc()

acc_vars = ["ACT", "CHE", "LCT", "STD", "TXP", "PPEGT", "AT", "OANCF"]

/Users/sondregrontvedt/Documents/Industriell Økonomi og Teknologiledelse/excel_extract_project/modelling/build_panel.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  return pd.concat(rows, ignore_index=True)


In [27]:
# ── 1. Missingness by year ─────────────────────────────────────────────────────
missing_by_year = (
    acc_data.groupby("Year")[acc_vars]
    .apply(lambda x: x.isnull().mean() * 100)
    .round(2)
)
print("=== Missing % by Year ===")
display(HTML(missing_by_year.to_html()))

=== Missing % by Year ===


,ACT,CHE,LCT,STD,TXP,PPEGT,AT,OANCF
Year,,,,,,,,
2005,2.17,7.43,2.79,53.25,40.56,50.46,0.93,0.93
2006,2.37,8.01,2.97,47.48,33.53,49.55,1.19,1.19
2007,1.43,7.14,1.71,45.71,27.71,51.71,0.29,0.86
2008,1.39,8.61,1.67,41.94,28.61,52.50,0.28,0.83
2009,1.59,8.49,1.86,44.83,32.89,50.13,0.53,0.80
2010,1.29,5.66,1.29,40.10,31.11,43.70,0.51,0.51
2011,1.23,5.16,1.47,40.29,28.50,27.03,0.25,1.23
2012,0.92,1.83,1.14,42.11,28.83,13.27,0.23,0.92
2013,0.89,2.00,1.11,39.56,26.22,7.56,0.22,0.22


In [28]:
# ── 2. Missingness by exchange ─────────────────────────────────────────────────
missing_by_exchange = (
    acc_data.groupby("Exchange")[acc_vars]
    .apply(lambda x: x.isnull().mean() * 100)
    .round(2)
)
print("=== Missing % by Exchange ===")
display(HTML(missing_by_exchange.to_html()))

=== Missing % by Exchange ===


,ACT,CHE,LCT,STD,TXP,PPEGT,AT,OANCF
Exchange,,,,,,,,
COX,2.58,3.72,2.64,35.14,21.79,22.61,1.01,0.19
HEX,1.38,1.08,1.38,27.95,26.77,13.54,0.10,0.15
ISX,0.76,0.76,0.76,28.24,53.44,18.70,0.76,1.91
OBX,0.14,4.10,0.14,33.95,25.22,14.99,0.14,0.24
STX,1.12,1.86,1.63,35.75,26.01,16.61,0.13,0.71


In [29]:
# ── 3. Missingness by variable (overall) ──────────────────────────────────────
missing_by_var = (acc_data[acc_vars].isnull().mean() * 100).round(2)
print("=== Missing % by Variable ===")
display(HTML(missing_by_var.to_frame("Missing %").to_html()))

=== Missing % by Variable ===


,Missing %
ACT,1.19
CHE,2.46
LCT,1.40
STD,33.52
TXP,26.04
PPEGT,16.68
AT,0.28
OANCF,0.45


In [35]:
# ── Critical variables ──────────────────────────────────────
critical_vars = ["OANCF", "AT", "ACT", "LCT"]
supplementary_vars = ["STD", "TXP", "PPEGT", "CHE"]

# Firms missing any critical variable
missing_critical = (
    acc_data.groupby("Ticker")[critical_vars]
    .apply(lambda x: x.isnull().sum())
    .reset_index()
)
missing_critical["Total Critical Missing"] = missing_critical[critical_vars].sum(axis=1)
missing_critical = missing_critical[missing_critical["Total Critical Missing"] > 0]
missing_critical = missing_critical.sort_values("Total Critical Missing", ascending=False)

print(f"Firms with missing critical variables: {len(missing_critical)}")
display(HTML(missing_critical.to_html(index=False)))

Firms with missing critical variables: 48


Ticker,OANCF,AT,ACT,LCT,Total Critical Missing
TRYG.CO,0,0,20,20,40
SAMPO.ST,0,0,20,20,40
SAMPO.HE,0,0,20,20,40
SAMPO.CO,0,0,20,20,40
ORRON.ST,0,0,0,19,19
SFAB.ST,0,0,8,8,16
NKT.CO,0,15,0,0,15
STEFb.ST,5,0,5,5,15
QLIRO.ST,1,0,6,6,13
MANTA.HE,0,0,5,5,10


In [36]:
insurance = acc_data[acc_data["Industry"] == "Insurance"]["Ticker"].nunique()
financials = acc_data[acc_data["Sector"] == "Financials"]["Ticker"].nunique()

print(f"Insurance firms (Industry): {insurance}")
print(f"Financials firms (Sector):  {financials}")

Insurance firms (Industry): 6
Financials firms (Sector):  6


In [37]:
# Firms missing any supplementary variable
missing_supplementary = (
    acc_data.groupby("Ticker")[supplementary_vars]
    .apply(lambda x: x.isnull().sum())
    .reset_index()
)
missing_supplementary["Total Supplementary Missing"] = missing_supplementary[supplementary_vars].sum(axis=1)
missing_supplementary = missing_supplementary[missing_supplementary["Total Supplementary Missing"] > 0]
missing_supplementary = missing_supplementary.sort_values("Total Supplementary Missing", ascending=False)

print(f"Firms with missing supplementary variables: {len(missing_supplementary)}")
display(HTML(missing_supplementary.to_html(index=False)))

Firms with missing supplementary variables: 566


Ticker,STD,TXP,PPEGT,CHE,Total Supplementary Missing
BIOPOR.CO,18,14,19,0,51
MGN.OL,18,18,10,5,51
AZT.OL,20,20,7,0,47
CRADb.ST,20,20,7,0,47
SINT.ST,20,20,7,0,47
PRICb.ST,19,20,7,0,46
ICP1V.HE,17,20,7,0,44
STZEb.ST,15,20,8,0,43
AAB.CO,19,16,7,1,43
SOFb.ST,20,17,6,0,43
